# J3-01 · Les 5 patterns multi-agents, et quand s'en passer

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 3 · créneau 08h30–09h45 · ⏱️ ~1h15 (dont 30 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Distinguer les 5 patterns de la documentation LangChain : **sous-agents**, **handoffs**, **skills**, **routeur**, **workflow sur mesure** · 2. Savoir quand le multi-agents est justifié, et quand c'est de la sur-ingénierie · 3. Comparer, sur le même help-desk, un **routeur** et des **sous-agents** |
| 🧱 **Fil rouge** | HelpDesk-Ops se dote de spécialistes : Procédures (KB), Diagnostic, Ticketing |
| ✅ **Prérequis** | Jours 1 et 2 ; KB indexée (cellule de la section 1) |
| 📦 **Versions** | `langchain 1.4` · `langgraph 1.2` · `langchain-qdrant 1.1` · `fastembed 0.9` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from helpdesk_ops import kb

# La KB du jour 3 (20 articles, data/kb/) est indexée dans Qdrant : détaillé en J3-02, ici on s'en sert
print("morceaux indexés :", kb.indexer())

## 2. Pourquoi plusieurs agents ?

### [Théorie] Le help-desk N1 qui dispatche vers les experts N2

*Analogie :* au téléphone, le N1 n'est pas expert réseau **et** expert ERP **et** expert comptes. Il qualifie, puis transmet au bon expert, qui ne reçoit que **le dossier utile**, pas toute la conversation.

Un agent unique avec 25 outils et un prompt de trois pages se dégrade : il choisit mal ses outils, son contexte déborde, et chaque nouveau besoin modifie un prompt dont tout dépend. Découper apporte :

| Bénéfice | Concrètement |
|---|---|
| **Isolation du contexte** | chaque spécialiste ne voit que sa tâche : prompt court, moins de confusion, moins de données personnelles qui circulent |
| **Moindre privilège** | un outil n'est exposé qu'au spécialiste qui en a besoin (OWASP ASI02) |
| **Évolution** | ajouter un spécialiste sans réécrire les autres (exercice 3.1) |
| **Observabilité** | une trace par spécialiste, imbriquée : on voit qui a fait quoi |

> ⚠️ **Le coût :** plus d'appels LLM (le coordinateur, puis chaque spécialiste), donc plus de latence. Mesuré pendant la préparation sur CPU : **4 à 8 minutes** pour une question à deux sujets avec un coordinateur, contre 20 à 90 secondes avec un routeur. Sur les GPU de la salle, l'écart reste du même ordre (en proportion).

## 3. Les 5 patterns (documentation LangChain, page *Multi-agent*)

| Pattern | Principe | Qui décide de la suite ? | Exemple HelpDesk-Ops |
|---|---|---|---|
| **Sous-agents** (*subagents*) | un coordinateur appelle des spécialistes **comme des outils** et garde la main | le coordinateur | « srv-erp-02 répond-il, et comment configurer le VPN ? » → Diagnostic **et** KB |
| **Handoffs** | un agent **passe la main** à un autre, qui poursuit la conversation | l'agent qui transmet | l'accueil transfère l'utilisateur au spécialiste Comptes, qui continue avec lui |
| **Skills** | un seul agent charge **à la demande** des instructions spécialisées | l'agent lui-même | l'agent charge la « fiche réflexe ERP » quand la question en parle |
| **Routeur** | une décision (règle ou LLM) envoie la demande à **un** spécialiste | le routeur, une seule fois | tri d'entrée : procédure, diagnostic ou ticket |
| **Workflow sur mesure** | un graphe explicite, avec des étapes déterministes et des agents là où il faut | le développeur | le jalon J2 : garde-fou → agent → qualification → validation |

> 💡 **Le « Tips » du Formateur :** ces patterns se **combinent**. Le jalon de ce soir est un *workflow sur mesure* (garde-fou de J1) qui contient un *coordinateur à sous-agents*. Retenez la question à se poser : **qui décide de la prochaine étape, et qui voit quel contexte ?**

## 4. Les spécialistes du jour

Le module `helpdesk_ops.multi_agents` fournit trois spécialistes, chacun un `create_agent` avec **ses** outils et **sa** consigne. On les construit une fois et on les réutilise dans les deux patterns comparés.

In [ ]:
import time

from helpdesk_ops.multi_agents import CONSIGNES, construire_routeur, construire_specialistes

specialistes = construire_specialistes(llm)
for nom in specialistes:
    print(f"{nom:<11} → {CONSIGNES[nom][:90]}…")

## 5. Pattern « routeur » : une décision, un spécialiste

### [Théorie] Le moins cher… tant que la demande n'a qu'un sujet

Une sortie structurée (J2-03) choisit **un** spécialiste ; le graphe y va avec `Command(goto=…)`. Un appel LLM de décision, puis le spécialiste : c'est le pattern le plus économique.

In [ ]:
routeur = construire_routeur(llm, specialistes)

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{routeur.get_graph().draw_mermaid()}\n```"))

In [ ]:
DEMANDES = [
    "Outlook plante dès que j'ouvre une pièce jointe",
    "srv-erp-02 répond-il au ping ?",
    "srv-erp-02 répond-il au ping ? Et comment je configure le VPN depuis l'hôtel ?",   # deux sujets
]


async def router_et_mesurer(demande: str) -> None:
    debut = time.perf_counter()
    with propagate_attributes(session_id="j3-01-routeur", tags=["jour-3", "routeur"]):
        etat = await routeur.ainvoke({"messages": [("user", demande)]},
                                     config={"callbacks": [langfuse_handler], "run_name": "routeur", "recursion_limit": 15})
    print(f"⏱ {time.perf_counter() - debut:5.0f} s | ❓ {demande}\n   🤖 {etat['messages'][-1].content[:220]}\n")


for demande in DEMANDES:
    await router_et_mesurer(demande)
langfuse.flush()

**👀 Ce que vous devez observer :**
- dans Langfuse, chaque trace `routeur` montre le nœud `router` puis **un seul** spécialiste ;
- la troisième demande, à **deux sujets**, n'est traitée qu'à moitié : le routeur n'a qu'un choix. C'est la limite structurelle du pattern ;
- ⚠️ **un routeur se trompe aussi.** Pendant la préparation, « Outlook plante… » est parti chez le **Diagnostic** (qui a vérifié `srv-mail-01`) au lieu de la KB ; et la réponse à la question double affirmait « aucun historique d'alerte dans les 24 h », une information qu'aucun outil n'avait fournie. Les deux se **mesurent** en J4 (justesse du routage, fidélité aux sources).

## 6. Pattern « handoff » : passer la main

### [Théorie] `Command(graph=Command.PARENT, goto=…)`

Dans un handoff, un agent **transfère** la conversation à un autre agent, qui continue directement avec l'utilisateur. Techniquement, un nœud (ou un outil) renvoie un `Command` qui désigne le prochain agent dans le **graphe parent**. On l'illustre sans LLM : la mécanique compte plus que le contenu.

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


def accueil(state: MessagesState) -> Command:
    demande = state["messages"][-1].content.lower()
    if "compte" in demande:   # l'accueil reconnaît un sujet qui n'est pas le sien : il passe la main
        return Command(goto="comptes", update={"messages": [AIMessage("Je vous transfère au spécialiste Comptes.")]})
    return Command(goto=END, update={"messages": [AIMessage("Je m'en occupe.")]})


def comptes(state: MessagesState) -> dict:
    return {"messages": [AIMessage("Spécialiste Comptes : votre compte est verrouillé, il se débloque dans 15 min.")]}


b = StateGraph(MessagesState)
b.add_node("accueil", accueil, destinations=("comptes", END))
b.add_node("comptes", comptes)
b.add_edge(START, "accueil")
b.add_edge("comptes", END)
handoff = b.compile()

for m in handoff.invoke({"messages": [("user", "Mon compte est bloqué")]})["messages"]:
    print(f"{type(m).__name__:<13} {m.content}")

**👀 Ce que vous devez observer :** l'accueil ne répond pas sur le fond, il **transfère** ; le spécialiste Comptes prend la suite **dans la même conversation**. Différence avec les sous-agents : ici, l'accueil ne reprend pas la main pour synthétiser.

Quand l'agent qui transfère est lui-même un sous-graphe (un `create_agent` dans un graphe plus large), il renvoie `Command(graph=Command.PARENT, goto="comptes")` pour désigner un nœud du graphe **parent**.

## 7. Multi-agents ou sur-ingénierie ?

| Signal | Plutôt un agent unique (ou un workflow) | Plutôt plusieurs agents |
|---|---|---|
| Nombre d'outils | moins d'une dizaine, d'un même domaine | plusieurs domaines, outils sensibles à isoler |
| Contexte | court, homogène | des sous-tâches qui n'ont pas besoin de tout voir |
| Latence acceptable | quelques secondes | un traitement de fond peut attendre |
| Équipes | un seul propriétaire | plusieurs équipes, chacune son spécialiste |
| Audit | peu d'exigences | savoir *qui* a décidé *quoi* (une trace par spécialiste) |

> 🛡️ **OWASP ASI08 (Cascading Failures) :** plus d'agents, c'est plus de points de défaillance. Un spécialiste qui tombe ne doit pas faire tomber le coordinateur : les protections de J2 (timeout, repli) s'appliquent **à chaque délégation**.

**Question pour la salle :** pour le help-desk de NordTech, le multi-agents est-il justifié ? Défendez une position, chiffres de latence à l'appui.

## 🧭 Récap

| Vous savez maintenant… | |
|---|---|
| Nommer les 5 patterns | sous-agents · handoffs · skills · routeur · workflow sur mesure |
| Les distinguer | qui décide de la suite, qui voit quel contexte |
| Router avec une sortie structurée | `construire_routeur` : décision `Literal`, puis `Command(goto=…)` |
| Passer la main | `Command(goto="autre_agent")` (ou `graph=Command.PARENT` depuis un sous-graphe) |
| Arbitrer | coût en latence contre isolation, moindre privilège, évolutivité |

**Prochain notebook, J3-02 :** le RAG sur Qdrant, puis le pattern sous-agents complet.

### ✅ Quiz express

1. Quelle différence entre sous-agents et handoffs ?
2. Pourquoi un routeur ne traite-t-il qu'à moitié une demande à deux sujets ?
3. Citez deux bénéfices de l'isolation du contexte.
4. Quel est le principal coût du multi-agents ?

<details>
<summary>Réponses</summary>

1. Avec des sous-agents, le coordinateur appelle les spécialistes comme des outils et garde la main pour synthétiser ; avec un handoff, l'agent transfère la conversation, et c'est le suivant qui continue.
2. Il prend une seule décision et n'envoie la demande qu'à un spécialiste.
3. Prompts plus courts et moins de confusion ; moins de données (personnelles) qui circulent ; outils limités au besoin.
4. La latence (et le coût) : chaque délégation ajoute des appels LLM.
</details>